In [1]:
# Библиотеки, которые пригодятся для решения


import cvxpy as cvx
import itertools
import heapdict as hd
import numpy as np
from numpy import unravel_index

# Задание на программирование [5 баллов]

Твоя задача --- реализовать алгоритм Branch and Bound для решения задачи судоку. Для удобства мы декомпозировали задачу на несколько функций, которые тебе необхоимо реализовать. Пожалуйста, не меняй имена функций. Если необходимо, ты можешь создавать дополнительные вспомогательные функции.

**Пример CVXPY**

В этой задаче тебе нужно будет использовать библиотеку CVXPY. Пример использования библиотеки можно найти в [Google Colab](https://colab.research.google.com/drive/13pCE4_T0UU1Ztfbsi0nSkY3PpOONZBwP?usp=sharing)

**INPUT description**

Твой алгоритм получит на вход один аргумент: судоку в виде двумерного списка 9 x 9, в котором нули обозначают пустые клетки. Например,

```
puzzle = [[4, 8, 0, 3, 0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0, 0, 0, 7, 1],
          [0, 2, 0, 0, 0, 0, 0, 0, 0],
          [7, 0, 5, 0, 0, 0, 0, 6, 0],
          [0, 0, 0, 2, 0, 0, 8, 0, 0],
          [0, 0, 0, 0, 0, 0, 0, 0, 0],
          [0, 0, 1, 0, 7, 6, 0, 0, 0],
          [3, 0, 0, 0, 0, 0, 4, 0, 0],
          [0, 0, 0, 0, 5, 0, 0, 0, 0]]
```

**OUTPUT description**

Твой алгоритм должен вернуть две переменные.

`solved_puzzle`

Решение задачи судоку. Например,
```
solved_puzzle = [[4, 8, 7, 3, 1, 2, 6, 9, 5],
                 [5, 9, 3, 6, 8, 4, 2, 7, 1],
                 [1, 2, 6, 5, 9, 7, 3, 8, 4],
                 [7, 3, 5, 8, 4, 9, 1, 6, 2],
                 [9, 1, 4, 2, 6, 5, 8, 3, 7],
                 [2, 6, 8, 7, 3, 1, 5, 4, 9],
                 [8, 5, 1, 4, 7, 6, 9, 2, 3],
                 [3, 7, 9, 1, 2, 8, 4, 5, 6],
                 [6, 4, 2, 9, 5, 3, 7, 1, 8]]
```

`constraints`

Ограничения, добавление которых делает LP-релаксацию задачи судоку целочисленной. Например, ограничения вида

```
constraints = [(8, 5, 2, 1), (5, 3, 4, 1)]
```

говорят, что добавление дополнительных ограничений $(x_{8,5})_{2} = 1$ and $(x_{5,3})_{4} = 1$ позволяют решить задачу судоку с помощью симплекс алгоритма.

Подробнее постановка задачи описана в pdf-условии.

## Основная функция

Решение судоку происходит с помощью функции `solve_sudoku`. Для ее реализации тебе понадобятся вспомогательные функции, которые определены ниже (см. раздел **Вспомогательные функции**).

После того как ты реализуешь алгоритм, ты можешь протестировать его на нескольких тестах (см. раздел **Тестирование**).

In [2]:
def solve_sudoku(puzzle):
    """
    Основная функция решения судоку с помощью алгоритма Branch and Bound.

    Args:
    puzzle: Массив 9x9 с цифрами от 0 до 9. Цифра 0 означает пустую клетку.

    Returns:
    Tuple (solved_puzzle, constraints), где:
    - solved_puzzle: Полностью заполненный массив 9x9 --- решение судоку.
    - constraints: Список дополнительных ограничений, которые делают
                   LP-релаксацию задачи судоку целочисленной.
    """

    # Удобнее работать с numpy-репрезентацией пазла
    puzzle_np = np.array(puzzle)

    # Создаем очередь с приоритетами (ключ: значение)
    pq = hd.heapdict()

    # Глобальный флаг-счетчик для очереди
    global_flag = 0

    # Решаем LP-релаксацию задачи без доп. ограничений
    current_const = []
    answer, value, index, flag = solve_sudoku_auxiliary(puzzle_np, [])
    
    # Добавляем счетчик итераций
    iteration = 0
    
    # Пока не найдем целочисленное решение, идем по циклу
    while True:
        iteration += 1
        print("Iteration:", iteration)
        
        # 1. Если текущее решение получилось целочисленным, возвращаем ответ
        if flag:
            return answer.tolist(), current_const  # Преобразуем numpy-массив в список и возвращаем ограничения
        
        # 2. Проверяем, есть ли переменная для ветвления
        if index is None:
            raise ValueError("Branching index is None, but further branching is needed. Check LP solution.")
        
        # 3. Делаем branch по одной переменной
        # index имеет вид (row, col, layer) где layer соответствует цифре (layer+1)
        current_const_0 = current_const + [list(index) + [0]]
        current_const_1 = current_const + [list(index) + [1]]
        
        # 4. Решаем LP-релаксации с новыми ограничениями и добавляем их в очередь
        try:
            answer_0, value_0, index_0, flag_0 = solve_sudoku_auxiliary(puzzle_np, current_const_0)
            if value_0 is not None:
                pq[global_flag] = (value_0, current_const_0, answer_0, index_0, flag_0)
                global_flag += 1
        except Exception as e:
            pass  # Пропускаем ветку, если LP не имеет решения
        
        try:
            answer_1, value_1, index_1, flag_1 = solve_sudoku_auxiliary(puzzle_np, current_const_1)
            if value_1 is not None:
                pq[global_flag] = (value_1, current_const_1, answer_1, index_1, flag_1)
                global_flag += 1
        except Exception as e:
            pass  # Пропускаем ветку, если LP не имеет решения
        
        # 5. Выбираем следующего кандидата для исследования
        if len(pq) == 0:
            raise ValueError("Не удалось найти решение!")
        
        candidate = pq.popitem()[1]
        value, current_const, answer, index, flag = candidate

## Вспомогательные функции

In [3]:
def generate_constraints(puzzle, variables):
    """
    Задает ограничения для задачи судоку, используя cvxpy.

    Args:
        puzzle (numpy.ndarray): 9x9 массив с цифрами от 0 до 9 (0 означает пустую клетку).
        variables (list of cvxpy.Variable): Список из 9 матриц бинарных переменных.

    Returns:
        list: Список ограничений (cvxpy constraints), которые задают правила судоку.
    """
    constraints = []

    # Ограничения: в каждой строке число встречается один раз
    for k in range(9):
        for i in range(9):
            constraints.append(cvx.sum(variables[k][i, :]) == 1)

    # Ограничения: в каждом столбце число встречается один раз
    for k in range(9):
        for j in range(9):
            constraints.append(cvx.sum(variables[k][:, j]) == 1)

    # Ограничения: в каждом 3x3 блоке число встречается один раз
    for k in range(9):
        for i, j in itertools.product([0, 3, 6], [0, 3, 6]):
            constraints.append(cvx.sum([variables[k][i + m, j + l] 
                                        for m, l in itertools.product(range(3), range(3))]) == 1)
    
    # Добавляем ограничение: в каждой ячейке должна быть ровно одна цифра
    for i in range(9):
        for j in range(9):
            constraints.append(sum(variables[k][i, j] for k in range(9)) == 1)

    # Фиксируем известные числа
    for k in range(9):
        lb = np.zeros((9, 9))
        lb[puzzle == (k + 1)] = 1
        constraints.append(variables[k] >= lb)

    return constraints

In [4]:
def solve_sudoku_auxiliary(puzzle, const):
    """
    Вспомогательная функция для решения LP-релаксации задачи судоку. В случае,
    если решение LP-релаксации не дает целочисленного решения, функция определяет
    переменную, по которой можно сделать разбиение.

    Args:
    puzzle: Массив 9x9 с цифрами от 0 до 9. Цифра 0 означает пустую клетку.
    const: Список дополнительных ограничений вида [row, column, value], которые
           должны быть дополнительно применены к судоку.

    Returns:
    answer: Массив 9x9 с цифрами --- частичное или полное решение судоку.
    value: значение целевой функции LP-релаксации.
    index: Tuple (row, column, layer), который определяет, по какой переменной
           мы сделаем разбиение в следующей итерации.
    flag: Бинарный индикатор того, что судоку решено полностью (True) или
          алгоритму требуется дальнейший бренчинг (False).
    """
    # Определяем переменные LP-модели: 9 матриц размером 9x9
    variables = [cvx.Variable((9, 9), boolean=True) for _ in range(9)]
    
    # Определяем целевую функцию LP-релаксации
    objective = cvx.Minimize(cvx.sum(cvx.maximum(*[variables[i] for i in range(9)])))
    
    # Генерируем основные ограничения судоку
    lp_constraints = generate_constraints(puzzle, variables)
    
    # Добавляем дополнительные ограничения из Branch and Bound (BnB)
    for c in const:
        if len(c) == 3:
            row, col, value = c
            lp_constraints.append(variables[value - 1][row, col] == 1)
        elif len(c) == 4:
            row, col, value, branch = c
            if branch == 1:
                lp_constraints.append(variables[value - 1][row, col] == 1)
            else:
                lp_constraints.append(variables[value - 1][row, col] == 0)
    
    # Решаем LP-релаксацию
    prob = cvx.Problem(objective, lp_constraints)
    value = prob.solve()
    
    if prob.status == 'infeasible':
        return None, None, None, False
    
    # Проверяем, является ли решение целочисленным
    solution_matrix = np.zeros((9, 9), dtype=int)
    for i in range(9):
        solution_matrix[np.round(variables[i].value) == 1] = i + 1
    
    # Если все значения целочисленны, судоку решено
    if np.all(solution_matrix > 0):
        return solution_matrix, value, None, True
    
    # Найти переменную, ближайшую к 0.5 для разбиения
    tol = 1e-3  # Допустимая погрешность для определения, что значение не является "целым"
    fractional_values = []
    for k in range(9):
        for i in range(9):
            for j in range(9):
                v = variables[k].value[i, j]
                if v is None:
                    continue  # Пропускаем, если значение не определено
                # Если значение не близко к 0 и не близко к 1, считаем его дробным
                if v > tol and v < 1 - tol:
                    fractional_values.append((abs(v - 0.5), (i, j, k)))
            
    # Если найдены дробные переменные, выбираем ту, которая ближе всего к 0.5
    if fractional_values:
        index = min(fractional_values, key=lambda x: x[0])[1]
    else:
            # Если дробных переменных нет, ищем ячейку, где ни один вариант не близок к 1
        index = None
        for i in range(9):
            for j in range(9):
                # Проверяем, что для всех k значение меньше 1 - tol
                if all(variables[k].value[i, j] < 1 - tol for k in range(9)):
                    # Выбираем k с наибольшим значением в данной ячейке
                    k_best = max(range(9), key=lambda k: variables[k].value[i, j])
                    index = (i, j, k_best)
                    break
            if index is not None:
                break
            
    return solution_matrix, value, index, False

## Тестирование

Ты можешь протестировать свой код на нескольких открытых тестах с помощью функции `test`.

Во время оценивания твое решение будет дополнительно протестировано на закрытых тестах. Рекомендуем самостоятельно найти примеры судоку и проверить на них свое решение.


In [5]:
def test():
    """
    Функция для тестирования решения.
    """

    sudoku_puzzle = [[8, 5, 9, 6, 1, 2, 4, 3, 7],
            [7, 2, 0, 8, 5, 4, 1, 6, 9],
            [1, 6, 4, 3, 7, 9, 5, 2, 8],
            [9, 8, 6, 1, 4, 7, 3, 5, 2],
            [3, 7, 5, 2, 6, 8, 9, 1, 4],
            [2, 4, 1, 5, 9, 0, 7, 8, 6],
            [4, 3, 2, 9, 8, 1, 6, 7, 5],
            [6, 1, 7, 0, 2, 5, 8, 9, 3],
            [5, 9, 8, 7, 3, 6, 2, 4, 1]]

    sudoku_sol = [[8, 5, 9, 6, 1, 2, 4, 3, 7],
            [7, 2, 3, 8, 5, 4, 1, 6, 9],
            [1, 6, 4, 3, 7, 9, 5, 2, 8],
            [9, 8, 6, 1, 4, 7, 3, 5, 2],
            [3, 7, 5, 2, 6, 8, 9, 1, 4],
            [2, 4, 1, 5, 9, 3, 7, 8, 6],
            [4, 3, 2, 9, 8, 1, 6, 7, 5],
            [6, 1, 7, 4, 2, 5, 8, 9, 3],
            [5, 9, 8, 7, 3, 6, 2, 4, 1]]

    print('=== solve_sudoku')
    print(' + Expected value: {}'.format((sudoku_sol, [])))
    print(' +     Your value: {}'.format(solve_sudoku(sudoku_puzzle)))
    
    sudoku_puzzle = [
            [5, 3, 0, 0, 7, 0, 0, 0, 0],
            [6, 0, 0, 1, 9, 5, 0, 0, 0],
            [0, 9, 8, 0, 0, 0, 0, 6, 0],
            [8, 0, 0, 0, 6, 0, 0, 0, 3],
            [4, 0, 0, 8, 0, 3, 0, 0, 1],
            [7, 0, 0, 0, 2, 0, 0, 0, 6],
            [0, 6, 0, 0, 0, 0, 2, 8, 0],
            [0, 0, 0, 4, 1, 9, 0, 0, 5],
            [0, 0, 0, 0, 8, 0, 0, 7, 9]]

    sudoku_sol = [
            [5, 3, 4, 6, 7, 8, 9, 1, 2],
            [6, 7, 2, 1, 9, 5, 3, 4, 8],
            [1, 9, 8, 3, 4, 2, 5, 6, 7],
            [8, 5, 9, 7, 6, 1, 4, 2, 3],
            [4, 2, 6, 8, 5, 3, 7, 9, 1],
            [7, 1, 3, 9, 2, 4, 8, 5, 6],
            [9, 6, 1, 5, 3, 7, 2, 8, 4],
            [2, 8, 7, 4, 1, 9, 6, 3, 5],
            [3, 4, 5, 2, 8, 6, 1, 7, 9]]

    print('=== solve_sudoku')
    print(' + Expected value: {}'.format((sudoku_sol, [])))
    print(' +     Your value: {}'.format(solve_sudoku(sudoku_puzzle)))

    sudoku_puzzle = [[8, 5, 0, 0, 0, 2, 4, 0, 0],
            [7, 2, 0, 0, 0, 0, 0, 0, 9],
            [0, 0, 4, 0, 0, 0, 0, 0, 0],
            [0, 0, 0, 1, 0, 7, 0, 0, 2],
            [3, 0, 5, 0, 0, 0, 9, 0, 0],
            [0, 4, 0, 0, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 8, 0, 0, 7, 0],
            [0, 1, 7, 0, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 3, 6, 0, 4, 0]]

    sudoku_sol = [[8, 5, 9, 6, 1, 2, 4, 3, 7],
            [7, 2, 3, 8, 5, 4, 1, 6, 9],
            [1, 6, 4, 3, 7, 9, 5, 2, 8],
            [9, 8, 6, 1, 4, 7, 3, 5, 2],
            [3, 7, 5, 2, 6, 8, 9, 1, 4],
            [2, 4, 1, 5, 9, 3, 7, 8, 6],
            [4, 3, 2, 9, 8, 1, 6, 7, 5],
            [6, 1, 7, 4, 2, 5, 8, 9, 3],
            [5, 9, 8, 7, 3, 6, 2, 4, 1]]

    print('=== solve_sudoku')
    print(' + Expected value: {}'.format((sudoku_sol, [])))
    print(' +     Your value: {}'.format(solve_sudoku(sudoku_puzzle)))

    sudoku_puzzle = [[4, 8, 0, 3, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0, 0, 0, 7, 1],
            [0, 2, 0, 0, 0, 0, 0, 0, 0],
            [7, 0, 5, 0, 0, 0, 0, 6, 0],
            [0, 0, 0, 2, 0, 0, 8, 0, 0],
            [0, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 0, 1, 0, 7, 6, 0, 0, 0],
            [3, 0, 0, 0, 0, 0, 4, 0, 0],
            [0, 0, 0, 0, 5, 0, 0, 0, 0]]

    sudoku_sol = [[4, 8, 7, 3, 1, 2, 6, 9, 5],
            [5, 9, 3, 6, 8, 4, 2, 7, 1],
            [1, 2, 6, 5, 9, 7, 3, 8, 4],
            [7, 3, 5, 8, 4, 9, 1, 6, 2],
            [9, 1, 4, 2, 6, 5, 8, 3, 7],
            [2, 6, 8, 7, 3, 1, 5, 4, 9],
            [8, 5, 1, 4, 7, 6, 9, 2, 3],
            [3, 7, 9, 1, 2, 8, 4, 5, 6],
            [6, 4, 2, 9, 5, 3, 7, 1, 8]]

    print('=== solve_sudoku')
    print(' + Expected value: {}'.format((sudoku_sol, [(6, 3, 2, 0)])))
    print(' +     Your value: {}'.format(solve_sudoku(sudoku_puzzle)))

    sudoku_puzzle = [
            [8, 0, 0, 0, 0, 0, 0, 0, 0],
            [0, 0, 3, 6, 0, 0, 0, 0, 0],
            [0, 7, 0, 0, 9, 0, 2, 0, 0],
            [0, 5, 0, 0, 0, 7, 0, 0, 0],
            [0, 0, 0, 0, 4, 5, 7, 0, 0],
            [0, 0, 0, 1, 0, 0, 0, 3, 0],
            [0, 0, 1, 0, 0, 0, 0, 6, 8],
            [0, 0, 8, 5, 0, 0, 0, 1, 0],
            [0, 9, 0, 0, 0, 0, 4, 0, 0]]
        
    sudoku_sol = [
            [8, 1, 2, 7, 5, 3, 6, 4, 9],
            [9, 4, 3, 6, 8, 2, 1, 7, 5],
            [6, 7, 5, 4, 9, 1, 2, 8, 3],
            [1, 5, 4, 2, 3, 7, 8, 9, 6],
            [3, 6, 9, 8, 4, 5, 7, 2, 1],
            [2, 8, 7, 1, 6, 9, 5, 3, 4],
            [5, 2, 1, 9, 7, 4, 3, 6, 8],
            [4, 3, 8, 5, 2, 6, 9, 1, 7],
            [7, 9, 6, 3, 1, 8, 4, 5, 2]]

    print('=== solve_sudoku')
    print(' + Expected value: {}'.format((sudoku_sol, [])))
    print(' +     Your value: {}'.format(solve_sudoku(sudoku_puzzle)))
test()

=== solve_sudoku
 + Expected value: ([[8, 5, 9, 6, 1, 2, 4, 3, 7], [7, 2, 3, 8, 5, 4, 1, 6, 9], [1, 6, 4, 3, 7, 9, 5, 2, 8], [9, 8, 6, 1, 4, 7, 3, 5, 2], [3, 7, 5, 2, 6, 8, 9, 1, 4], [2, 4, 1, 5, 9, 3, 7, 8, 6], [4, 3, 2, 9, 8, 1, 6, 7, 5], [6, 1, 7, 4, 2, 5, 8, 9, 3], [5, 9, 8, 7, 3, 6, 2, 4, 1]], [])
Iteration: 1
 +     Your value: ([[8, 5, 9, 6, 1, 2, 4, 3, 7], [7, 2, 3, 8, 5, 4, 1, 6, 9], [1, 6, 4, 3, 7, 9, 5, 2, 8], [9, 8, 6, 1, 4, 7, 3, 5, 2], [3, 7, 5, 2, 6, 8, 9, 1, 4], [2, 4, 1, 5, 9, 3, 7, 8, 6], [4, 3, 2, 9, 8, 1, 6, 7, 5], [6, 1, 7, 4, 2, 5, 8, 9, 3], [5, 9, 8, 7, 3, 6, 2, 4, 1]], [])
=== solve_sudoku
 + Expected value: ([[5, 3, 4, 6, 7, 8, 9, 1, 2], [6, 7, 2, 1, 9, 5, 3, 4, 8], [1, 9, 8, 3, 4, 2, 5, 6, 7], [8, 5, 9, 7, 6, 1, 4, 2, 3], [4, 2, 6, 8, 5, 3, 7, 9, 1], [7, 1, 3, 9, 2, 4, 8, 5, 6], [9, 6, 1, 5, 3, 7, 2, 8, 4], [2, 8, 7, 4, 1, 9, 6, 3, 5], [3, 4, 5, 2, 8, 6, 1, 7, 9]], [])
Iteration: 1
 +     Your value: ([[5, 3, 4, 6, 7, 8, 9, 1, 2], [6, 7, 2, 1, 9, 5, 3, 4, 8], 

Only problem- was not able to take constrains into account((